# Otomatik Hiperparametre Optimizasyonu: GridSearchCV, RandomizedSearchCV ve Optuna

Bu modül; makine öğrenmesi modellerinin genelleme başarımını maksimize etmek için kritik olan **hiperparametre optimizasyonu (Hyperparameter Tuning)** yöntemlerini inceler. Kaba kuvvet ızgara arama (`GridSearchCV`) ile modern Bayesyen optimizasyon çerçevesi (`Optuna`) arasındaki hız ve başarım farkını ampirik olarak kıyaslar.

---

## 1. Arama Stratejileri Karşılaştırması

1. **Izgara Arama (Grid Search):** Belirlenen parametre uzayındaki tüm olası kartezyen kombinasyonları test eder. Parametre sayısı arttığında hesaplama maliyeti üstel olarak patlar ($O(N^d)$).
2. **Rastgele Arama (Random Search):** Parametre uzayından rassal dağılımlar ile belirli sayıda örnek çeker. Sürekli uzaylarda ızgara aramadan çok daha etkilidir (Bergstra & Bengio, 2012).
3. **Bayesyen Optimizasyon (Optuna - TPE):**
   Önceki denemelerin sonuçlarını kullanarak bir olasılık dağılım modeli kurar. **Tree-structured Parzen Estimator (TPE)** algoritması ile başarı getiren bölgelerden daha yoğun, başarısız bölgelerden daha seyrek örnekleme yapar. Kötü giden denemeleri erken durdurur (Pruning).


In [ ]:
import numpy as np
import optuna
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, f1_score

# 1. Benchmark Sınıflandırma Veri Kümesi
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Eğitim Örneği: {len(X_train)}, Test Örneği: {len(X_test)}")


## 2. Geleneksel Yöntem: GridSearchCV ile Arama

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 8],
    'min_samples_split': [2, 5],
    'criterion': ['gini', 'entropy']
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_grid_model = grid_search.best_estimator_
y_pred_grid = best_grid_model.predict(X_test)

print(f"GridSearch En İyi Parametreler: {grid_search.best_params_}")
print(f"GridSearch Test F1-Score     : {f1_score(y_test, y_pred_grid):.4f}")


## 3. Modern Yöntem: Optuna ile Bayesyen Optimizasyon (TPE)

In [ ]:
# Optuna günlük kayıtlarını sessize alma
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    # Denenecek hiperparametre aralıkları
    n_estimators = trial.suggest_int('n_estimators', 30, 200)
    max_depth = trial.suggest_int('max_depth', 2, 15)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    criterion = trial.suggest_categorical('criterion', ['gini', 'entropy'])
    max_features = trial.suggest_float('max_features', 0.2, 1.0)
    
    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        criterion=criterion,
        max_features=max_features,
        random_state=42,
        n_jobs=-1
    )
    
    # 5-Katlı Çapraz Doğrulama ile Ortalama F1 Skoru
    score = cross_val_score(clf, X_train, y_train, cv=5, scoring='f1').mean()
    return score

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=35)

print(f"Optuna En İyi Deneme F1 Skoru : {study.best_value:.4f}")
print("Optuna En İyi Parametreler    :")
for k, v in study.best_params.items():
    print(f"  - {k}: {v}")

# En iyi Optuna modeli ile test doğrulaması
best_optuna_rf = RandomForestClassifier(**study.best_params, random_state=42)
best_optuna_rf.fit(X_train, y_train)
y_pred_optuna = best_optuna_rf.predict(X_test)
print(f"Optuna Nihai Test F1-Skoru    : {f1_score(y_test, y_pred_optuna):.4f}")


## 4. Mühendislik Çıkarımları ve Hız Karşılaştırması

1. **Örnekleme Verimliliği:** GridSearchCV tüm $3 \times 3 \times 2 \times 2 = 36$ sabit noktayı tararken, Optuna sürekli aralıklarda float değerleri de hesaba katarak en iyi çözüme çok daha az denemede ulaşır.
2. **Pruning (Budama):** Optuna ara adımlarda başarı vadetmeyen hiperparametre kombinasyonlarını (örn. ilk 2 epochta başarısız olan denemeleri) erken keserek donanım maliyetini %70'e kadar azaltır.
3. **Büyük Veri Kümesi Kuralı:** 3'ten fazla hiperparametre optimize edilecekse GridSearch yerine daima Bayesyen Optimizasyon (Optuna) tercih edilmelidir.
